[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/glass_cyclide/glass_cyclide.ipynb)

# A Glass Dupin Cyclide

A ring of glass stands on a checkered floor, its tube thick on one side and thin on the other, with a round bite taken out of its thick side and a red ball behind it, while a camera circles it. The ring is a Dupin cyclide, a torus inverted in a sphere, and the light bends through it, reflects off it and fades inside it.

In the conformal model of space a point is a null antivector, placed by a translator carrying the origin, `translator(offset) >> origin`, and a quadric maps a point to its polar sphere, `Sphere <- Point`; its pairing with a point, `point & quadric(point)`, is zero on the surface and negative inside, and at a point of the surface its sphere is the tangent sphere. In this model the torus, a quartic surface in coordinates, is a quadric, and a sphere inversion is a versor, so `inversion >> torus(inversion << Point)` is the cyclide. A ball is a quadric as well, and the bite is the ring intersected with a ball's outside, the ball's quadric negated. A ray meets a solid where the largest of its quadrics' pairings changes sign; the shader, written in numga, finds that change by stepping and halving, and follows each pixel's light through the glass.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from examples.util.animation import save_figure, show_animation
from examples.quadrics.glass_cyclide import render
from examples.util.wgsl import Shader, image_figure, supersampled

In [ ]:
from numga.algebras import CGA3D
# Space as the conformal model: x y z, and w and e, which square to one and minus one.
ga = CGA3D
mv = NumpyContext(ga).multivector
# Spheres and planes are vectors, planes through the origin those within x y z. Points are null antivectors, and
# directions of flat space the antivectors square to planes through the origin, those containing w and e. Motors move
# and turn space conformally.
Sphere = ga.gatype.vector()                                                    # Vector+
OriginPlane = Sphere.derive.within("x y z")                                    # Vector+
Point = ga.gatype.antivector()                                                 # Antivector-
Direction = Point.derive.containing("w e")                                     # Antivector-
Motor = ga.gatype.rotor()                                                      # Rotor+
# A camera only turns: a rotor of flat space.
Rotation = Motor.derive.within("x y z")                                        # Rotor+
Quadric = ga.gatype((Sphere, Point))
# The origin and the point at infinity; the plane at infinity, whose pairing with a point is its weight.
origin = (mv.w.dual() + mv.e.dual()) / 2                                       # [] Point-
infinity = mv.e.dual() - mv.w.dual()                                           # [] Point-
horizon = mv.e - mv.w                                                          # [] Sphere+

## 1. A torus inverted

Inverting the torus in a sphere off its centre swells the tube on the far side and narrows it on the near one. The torus's own points, carried by the same inversion, lie on the cyclide: the inversion carries the surface with them. A point's pairing scales with the square of its weight, its pairing with the plane at infinity, so `on_surface` divides it out.

In [ ]:
def sphere(centre: Point, radius: float) -> Sphere:
    """The sphere about a centre: positive inside, paired with a point."""
    return centre.dual_inverse() - radius**2 / 2 * horizon


def ball(surface: Sphere) -> Quadric:
    """The solid inside a sphere, or on a plane's negative side: its pairing times that of the plane at infinity,
    symmetrized."""
    return (surface * (horizon & Point) + horizon * (surface & Point)) / 2


def torus(radius: float, tube: float) -> Quadric:
    """The solid torus about the z axis: a core circle of `radius` in the xy plane, and a `tube` about it."""
    core_sphere = sphere(origin, np.hypot(radius, tube))
    return core_sphere * (core_sphere & Point) + radius**2 * (mv.z * (mv.z & Point) - tube**2 * horizon * (horizon & Point))


def translator(offset: Direction) -> Motor:
    """The motor carrying every point along `offset`, generated by the line joining it with the point at infinity."""
    return (-(infinity & offset).dual() / 2).exp()


def torus_points(radius: float, tube: float, n_around: int, n_across: int) -> Point:
    """Points on a torus's surface `[across, around]`: a point on the tube's circle turned about the core, then about
    z."""
    around = (mv.xy * (-np.linspace(0, 2 * np.pi, n_around, endpoint=False) / 2)).exp()      # [around] Rotation+
    across = (mv.xz * (-np.linspace(0, 2 * np.pi, n_across, endpoint=False)[:, None] / 2)).exp()  # [across, 1] Rotation+
    on_tube = across >> (translator(mv.x.dual() * tube) >> origin)                           # [across, 1] Point-
    return around >> (translator(mv.x.dual() * radius) >> on_tube)                           # [across, around] Point-


# The torus, the sphere it is inverted in, off its centre, and the points around and across its tube.
torus_radius, torus_tube = 2.5, 0.9
inversion_offset, inversion_radius = 8.0, 6.25
n_around, n_across = 24, 12

inversion = (translator(mv.x.dual() * inversion_offset) >> sphere(origin, inversion_radius)).normalized()   # [] Sphere+
# The cyclide: the solid torus inverted in the sphere, a versor, which carries a map as it carries anything.
surface = inversion >> torus(torus_radius, torus_tube)(inversion << Point)               # [] Sphere+ <- Point-
inverted = inversion >> torus_points(torus_radius, torus_tube, n_around, n_across)       # [across, around] Point-
on_surface = (inverted & surface(inverted)) / (horizon & inverted).squared()             # [across, around] Scalar+

## 2. The solids

The ring is stood up by a motor, its bounding centre carried to the origin, tilted about x, and lifted to rest on the floor; the bite is a ball moved with it. Each solid carries the share of light it lets through and its refractive index: the glass lets nearly all of it through, the ball and the floor none.

In [ ]:
# The ring's bounding centre and radius, its tilt about x, and how high it is lifted; the bite's centre and radius;
# the ball's; the floor's bounding radius; and the floor's tiles.
cyclide_centre, cyclide_radius, cyclide_tilt, lift = mv.x.dual() * 2.04, 2.6, np.radians(55.0), mv.z.dual() * 2.25
bite_centre, bite_radius = translator(mv.x.dual() * 0.6 + mv.y.dual() * 1.2 + mv.z.dual() * 0.9) >> origin, 0.9
ball_centre, ball_radius = translator(mv.x.dual() * 0.6 + mv.y.dual() * 3.4 + mv.z.dual() * 1.1) >> origin, 1.1
floor_radius, tiles = 60.0, 0.8

# The ring stood up: its bounding centre to the origin, tilted about x, lifted above the floor.
centring = translator(-cyclide_centre)                                         # [] Motor+
tilt = (mv.yz * (-cyclide_tilt / 2)).exp()                                     # [] Motor+
place = translator(lift) * tilt * centring                                     # [] Motor+
shapes = [
    # The glass cyclide, less the bite: outside the bite's ball.
    [place >> surface(place << Point), -ball(place >> sphere(bite_centre, bite_radius))],
    [ball(sphere(ball_centre, ball_radius))],
    # The floor: everything below the plane z = 0.
    [ball(-mv.z)],
]
counts = np.array([len(shape) for shape in shapes])
ends = np.cumsum(counts)
quadrics = stack([quadric.cast(Quadric) for shape in shapes for quadric in shape])          # [quadrics] Sphere+ <- Point-
# Each solid's bounding sphere, a little larger than the ball's own.
solids = dict(first=ends - counts, end=ends,
              centre=stack([translator(lift) >> origin, ball_centre, origin]),
              radius=np.array([cyclide_radius, ball_radius * 1.01, floor_radius]),
              colour=np.array([[0.9, 0.96, 0.97], [0.85, 0.22, 0.16], [0.88, 0.87, 0.85]]),
              transparency=np.array([0.96, 0.0, 0.0]), refraction=np.array([1.5, 1.0, 1.0]),
              across=stack([0 * mv.x, 0 * mv.x, mv.x * tiles]), along=stack([0 * mv.y, 0 * mv.y, mv.y * tiles]))

## 3. The shader

Each invocation follows one pixel's light. At glass it refracts by Snell's law, with the outward direction square to the surface read off the tangent sphere, and adds what the reflected ray glimpses, by Fresnel's share in Schlick's approximation; past the critical angle all of it reflects, and inside the glass it fades by the glass's tint per unit length. An opaque surface ends the path with its colour, lit by the lamp and shadowed by what lies towards it, glass shadowing less.

In [ ]:
source = supersampled() + """
// A solid: the intersection of its quadrics, `quadrics[first]` up to `quadrics[end]`, each negative
// inside it, within a bounding sphere about `centre`. Its colour, the tint glass takes on per unit
// length; the share of light it lets through, and its refractive index; two planes through the origin
// whose pairings with a point, in unit steps, lay a checker over it.
struct Solid {
    first: u32,
    end: u32,
    centre: Point,
    radius: f32,
    colour: vec3<f32>,
    transparency: f32,
    refraction: f32,
    across: OriginPlane,
    along: OriginPlane,
}

var<storage, read> solids: array<Solid>;
var<storage, read> quadrics: array<Quadric>;
var<uniform> turn: Rotation;
var<uniform> eye: Point;
var<uniform> lamp: Direction;
var<uniform> half_view: f32;
alias Sample = vec3<f32>;

const far = 1e6;
// The step a ray takes off a surface before it looks again; the steps across a bounding sphere, and the
// halvings of the one that crosses.
const clearance = 1e-3;
const n_steps = 40u;
const n_halvings = 18u;
const n_bounces = 8u;
// The look: light where the lamp does not reach, the light glass lets into its shadow, the checker's
// darkening, and the sky from the horizon up.
const ambient = 0.3;
const glass_shadow = 0.75;
const checker = 0.18;
const horizon_colour = vec3<f32>(0.86, 0.88, 0.92);
const zenith_colour = vec3<f32>(0.36, 0.5, 0.72);
// The sun seen in a reflection: its brightness, and how tightly it gathers about the lamp's direction.
const sun = 6.0;
const sun_size = 2000.0;

// The sphere about a centre: positive inside, paired with a point.
fn sphere(centre: Point, radius: f32) -> Sphere {
    return centre.dual_inverse() - radius * radius / 2 * horizon;
}

// How a point moves per unit distance as the translator along a unit direction carries it: its commutator with the
// translator's generator, the dual of the line joining the point at infinity with the direction.
fn velocity(start: Point, direction: Direction) -> Point {
    return start.commutator((infinity & direction).dual());
}

// The point a distance along a ray: its start carried by the translator along its unit direction,
// `translator(direction * distance) >> start`, written out for a start of unit weight: the start, plus the distance
// times its velocity, plus half the distance squared times the point at infinity. So written, the start's weight,
// which single precision holds only roughly for a far point, never meets the distance squared.
fn ahead(start: Point, direction: Direction, distance: f32) -> Point {
    return start + velocity(start, direction) * distance + infinity * (distance * distance / 2);
}

// How far a point lies outside a solid: the largest of its quadrics' forms, negative inside them all.
fn outside(solid: Solid, point: Point) -> f32 {
    var largest = -far;
    for (var index = solid.first; index < solid.end; index++) {
        largest = max(largest, point & quadrics[index](point));
    }
    return largest;
}

// The first distance along a ray at which it crosses a solid's surface, or `far`.
fn crossing(solid: Solid, origin: Point, direction: Direction) -> f32 {
    // The stretch of the ray inside the bounding sphere: how far along the ray the sphere's centre lies, the rate at
    // which the sphere's pairing with the start grows as the start moves along the ray, and the square of half the
    // chord, that squared plus twice the sphere's pairing with the start, which is the squared radius less the start's
    // squared distance from the centre.
    let bounding = sphere(solid.centre, solid.radius);
    let along = bounding & velocity(origin, direction);
    let half_chord_squared = along * along + 2 * (bounding & origin);
    let start = max(along - sqrt(max(half_chord_squared, 0.0)), clearance);
    let end = along + sqrt(max(half_chord_squared, 0.0));
    if (half_chord_squared <= 0 || end <= start) {
        return far;
    }
    let started_outside = outside(solid, ahead(origin, direction, start)) > 0;
    var before = start;
    for (var step = 1u; step <= n_steps; step++) {
        let after = start + (end - start) * f32(step) / f32(n_steps);
        if ((outside(solid, ahead(origin, direction, after)) > 0) != started_outside) {
            var low = before;
            var high = after;
            for (var halving = 0u; halving < n_halvings; halving++) {
                let middle = (low + high) / 2;
                if ((outside(solid, ahead(origin, direction, middle)) > 0) == started_outside) {
                    low = middle;
                } else {
                    high = middle;
                }
            }
            return high;
        }
        before = after;
    }
    return far;
}

struct Hit {
    distance: f32,
    solid: u32,
}

fn nearest(origin: Point, direction: Direction) -> Hit {
    var hit = Hit(far, 0u);
    for (var index = 0u; index < arrayLength(&solids); index++) {
        let distance = crossing(solids[index], origin, direction);
        if (distance < hit.distance) {
            hit = Hit(distance, index);
        }
    }
    return hit;
}

// A solid's outward normal at a point on it, as the unit plane through the origin parallel to the tangent plane, whose
// dual is the outward direction: from the tangent sphere of the quadric whose surface the point is on, the one the
// point lies least inside, that sphere flattened at the point, `tangent + (tangent | horizon) * point.dual_inverse()`,
// is the tangent plane, and its part within x y z the plane through the origin.
fn normal(solid: Solid, point: Point) -> OriginPlane {
    var surface = solid.first;
    for (var index = solid.first; index < solid.end; index++) {
        if ((point & quadrics[index](point)) > (point & quadrics[surface](point))) {
            surface = index;
        }
    }
    let tangent = quadrics[surface](point);
    return (tangent + (tangent | horizon) * point.dual_inverse()).cast(OriginPlane).normalized();
}

// The sky along a direction: by how far it looks up, its pairing with the floor's plane, and how squarely it looks
// into the sun, its pairing with the plane square to the lamp's direction.
fn sky(direction: Direction) -> vec3<f32> {
    let up = mv.z & direction;
    let sunward = lamp.dual_inverse() & direction;
    return mix(horizon_colour, zenith_colour, clamp(up, 0.0, 1.0)) + sun * pow(max(sunward, 0.0), sun_size);
}

// The light towards the lamp from a point: dimmed by each solid in the way, opaque ones fully.
fn sunlight(point: Point) -> f32 {
    var light = 1.0;
    for (var index = 0u; index < arrayLength(&solids); index++) {
        if (crossing(solids[index], point, lamp) < far) {
            light *= solids[index].transparency * glass_shadow;
        }
    }
    return light;
}

// An opaque surface's colour at a point: lit by the lamp, shadowed, and checkered.
fn surface(solid: Solid, point: Point, facing: OriginPlane) -> vec3<f32> {
    let lit = max(facing & lamp, 0.0) * sunlight(ahead(point, facing.dual(), clearance));
    let parity = abs((floor(solid.across & point) + floor(solid.along & point)) % 2.0);
    return solid.colour * (1 - checker * parity) * (ambient + (1 - ambient) * lit);
}

// What a ray glimpses without passing through glass: the surface it meets, or the sky.
fn glimpse(origin: Point, direction: Direction) -> vec3<f32> {
    let hit = nearest(origin, direction);
    if (hit.distance >= far) {
        return sky(direction);
    }
    let solid = solids[hit.solid];
    let point = ahead(origin, direction, hit.distance);
    let facing = normal(solid, point);
    return mix(surface(solid, point, facing), sky(direction), solid.transparency);
}

// The light along the ray through a place on the image: from the camera's own -z, turned by its rotor.
fn sample(place: vec2<f32>) -> Sample {
    let across = screen(place) * half_view;
    let toward = turn >> (-mv.z.dual() + (axes * across).sum());
    // A direction's length squared is its pairing with the plane square to it.
    var direction = toward / sqrt(toward.dual_inverse() & toward);
    var position = eye;
    var colour = vec3<f32>(0.0);
    var throughput = vec3<f32>(1.0);
    for (var bounce = 0u; bounce < n_bounces; bounce++) {
        let hit = nearest(position, direction);
        if (hit.distance >= far) {
            colour += throughput * sky(direction);
            break;
        }
        let solid = solids[hit.solid];
        position = ahead(position, direction, hit.distance);
        let outward = normal(solid, position);
        let entering = (outward & direction) < 0;
        // Light that crossed glass to get here faded by its tint, per unit length.
        throughput *= select(pow(solid.colour, vec3<f32>(hit.distance)), vec3<f32>(1.0), entering || solid.transparency == 0);
        let facing = select(-outward, outward, entering);
        colour += throughput * (1 - solid.transparency) * surface(solid, position, facing);
        // At glass the ray refracts, by Snell's law, and reflects in the plane along the surface; past the critical
        // angle all of it reflects.
        let cosine = -(facing & direction);
        let ratio = select(solid.refraction, 1 / solid.refraction, entering);
        let sine_squared = ratio * ratio * (1 - cosine * cosine);
        let reflected = facing >> direction;
        let refracted = direction * ratio + facing.dual() * (ratio * cosine - sqrt(max(1 - sine_squared, 0.0)));
        // Schlick's approximation to Fresnel's share of reflected light.
        let normal_reflectance = ((solid.refraction - 1) / (solid.refraction + 1)) * ((solid.refraction - 1) / (solid.refraction + 1));
        let reflectance = normal_reflectance + (1 - normal_reflectance) * pow(1 - cosine, 5.0);
        let total = sine_squared > 1;
        colour += throughput * solid.transparency * select(reflectance, 0.0, total) * glimpse(ahead(position, reflected, clearance), reflected);
        throughput *= solid.transparency * select(1 - reflectance, 1.0, total);
        direction = select(refracted, reflected, total);
        position = ahead(position, direction, clearance);
        if (max(throughput.x, max(throughput.y, throughput.z)) < 0.01) {
            break;
        }
    }
    return colour;
}
"""

## 4. The camera's view

A camera is a rotation, from one looking along its own -z with y up, and an eye; the image comes from `supersampled()`, a turned grid of samples in each pixel, averaged, four a side on a GPU and one on a CPU; the shader says what one sample sees, `fn sample(place: vec2<f32>) -> Sample`.

In [ ]:
# The lamp's elevation and azimuth; the target the cameras look at, their distance from it and elevation, and half
# the height of their view.
lamp_elevation, lamp_azimuth = np.radians(49.0), np.radians(-50.0)
target, distance, elevation, half_view = translator(mv.z.dual() * 1.9) >> origin, 11.0, np.radians(20.0), 0.31
# The still's pixels and the animation's, its frames, and the duration of a frame.
image_shape, animation_shape, n_frames, duration_ms = (450, 720), (330, 528), 96, 40


def cameras(azimuths: np.ndarray) -> tuple[Rotation, Point]:
    """Cameras circling the ring at the given azimuths, each turned to look at the target, and their eyes."""
    turns = (mv.xy * (-azimuths / 2)).exp() * (mv.yz * (-(np.pi / 2 - elevation) / 2)).exp()   # [cameras] Rotation+
    return turns, translator((turns >> mv.z.dual()) * distance) >> target


# The lamp's direction: x raised towards z by its elevation, then turned towards y by its azimuth.
lamp_turn = (mv.xy * (-lamp_azimuth / 2)).exp() * (mv.xz * (-lamp_elevation / 2)).exp()        # [] Rotation+
lamp = lamp_turn >> mv.x.dual()                                                                # [] Direction-
# The shader takes the point and the plane at infinity, and the directions across and up the camera's own frame, as
# constants.
tracer = Shader(source, ga, dict(Sphere=Sphere, OriginPlane=OriginPlane, Point=Point, Direction=Direction,
                                 Rotation=Rotation, Quadric=Quadric),
                dict(infinity=infinity, horizon=horizon, axes=stack((mv.x.dual(), mv.y.dual()))))
still, = render.frames(tracer, quadrics, solids, *cameras(np.zeros(1)), lamp, half_view, image_shape)
save_figure(image_figure(still), "glass_cyclide");

## 5. Around the ring

Circling the ring shows it edge on and face on: through the thin side the floor bends little, through the thick side it folds over, and the bite shows as a lens cut into the glass.

In [ ]:
azimuths = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
show_animation(render.frames(tracer, quadrics, solids, *cameras(azimuths), lamp, half_view, animation_shape),
               "glass_cyclide", duration_ms)

In [ ]:
# checks
# The torus's points, inverted, lie on the cyclide.
np.testing.assert_allclose(on_surface.kernel, 0, atol=1e-10)